# Notebook 05 — Model Development

## Purpose

Compare classifiers on the 1,199 manually labeled comments. The four tasks are trained and scored separately:

- `policy_relevance`
- `feedback_type`
- `topic`
- `sentiment`

`actor` is stored in the annotation file and is not a task in this notebook.

## Models

Each task is given the same five candidates:

1. Multinomial naive Bayes on TF-IDF word and bigram features
2. Complement naive Bayes on the same features
3. Class-weighted logistic regression on the same features
4. Class-weighted linear support-vector machine on the same features
5. Class-weighted logistic regression on the multilingual MiniLM embeddings saved by notebook 04

## Selection rule

The 1,199 rows are split, within each task, into train 70 percent, validation 10 percent, and test 20 percent. The split is stratified and uses seed 42. The vectorizer and the scaler are fit on the training rows only.

The selected model is the candidate with the highest validation macro-F1. If two candidates are tied to four decimal places, the higher validation weighted-F1 is used. The test split is scored once, after that choice, and only for the selected model. Test labels are not used to choose the model.

These test scores are the held-out result. They are not the scores of a model that has been refit on all 1,199 rows.

## Text

The model text is `comment_whitespace_normalized` from the raw working corpus, joined by `record_key`. The same field exists for all 5,292 comments. The 25 differences between annotation text and raw text remain in the mismatch register. No label is renamed or merged.

## Language

The TF-IDF models use letter tokens of length two or more. They do not use an English or Somali stopword list. The embedding model is `paraphrase-multilingual-MiniLM-L12-v2`. A score from that model does not show that the embedding understands Somali.


## 1. Prepare the labeled comments

Join each annotated row to the raw comment text and to its saved embedding.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

candidates = [Path.cwd(), Path.cwd().parent, Path(r"D:\NIRA_CASE_STUDY")]
ROOT = None
for candidate in candidates:
    if (candidate / "NIRA_Facebook_Datasets_5292.csv").exists():
        ROOT = candidate.resolve()
        break
if ROOT is None:
    raise FileNotFoundError("Project root not found.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.supervised import (
    CLASSICAL_MODEL_NAMES,
    EMBEDDING_MODEL_NAME,
    make_classical_pipeline,
    make_embedding_estimator,
)
from config.project_config import (
    CLASSIFICATION_TARGETS,
    EXPECTED_ANNOTATION_N,
    EXPECTED_RAW_N,
    RANDOM_SEED,
)

PREP = ROOT / "outputs" / "01_data_preparation" / "20261018_content_review"
TOPIC = ROOT / "outputs" / "04_topic_modeling" / "20261019_topic_models"
RUN_ID = "20261019_supervised"
OUT = ROOT / "outputs" / "05_model_development" / RUN_ID
OUT.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(PREP / "raw_corpus_working.csv").sort_values("source_row").reset_index(drop=True)
ann = pd.read_csv(PREP / "annotation_linked.csv")
embeddings = np.load(TOPIC / "comment_embeddings.npy")
if len(raw) != EXPECTED_RAW_N or len(ann) != EXPECTED_ANNOTATION_N:
    raise AssertionError("Unexpected corpus or annotation size.")
if embeddings.shape[0] != len(raw):
    raise AssertionError("Embedding rows do not match the working corpus.")
if not set(ann["record_key"]).issubset(set(raw["record_key"])):
    raise AssertionError("An annotation key is missing from the raw corpus.")

text_by_key = raw.set_index("record_key")["comment_whitespace_normalized"].astype(str)
emb_index = pd.Series(np.arange(len(raw)), index=raw["record_key"])
ann = ann.copy()
ann["model_text"] = ann["record_key"].map(text_by_key)
ann["emb_row"] = ann["record_key"].map(emb_index)
if ann["model_text"].isna().any() or ann["emb_row"].isna().any():
    raise AssertionError("Annotation rows did not join to model text.")
ann["emb_row"] = ann["emb_row"].astype(int)
print("Annotated rows:", len(ann))
print("Tasks:", ", ".join(CLASSIFICATION_TARGETS))
print("Output directory:", OUT)


Annotated rows: 1199
Tasks: policy_relevance, feedback_type, topic, sentiment
Output directory: D:\NIRA_CASE_STUDY\outputs\05_model_development\20261019_supervised


### Interpretation

The four tasks stay separate, and `actor` is not trained. Each annotated comment uses the raw working-corpus text, so the same field can later be scored for all 5,292 comments. The 25 text differences recorded earlier are not overwritten.


## 2. Compare five models on each task

Select the model with the highest validation macro-F1. Score the test split only after that choice.


In [2]:
def score_labels(y_true, y_pred, labels):
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=labels, zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", labels=labels, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }

validation_rows = []
selected = {}
test_rows = []
split_rows = []
reports = {}

for task in CLASSIFICATION_TARGETS:
    y = ann[task].astype(str)
    labels = sorted(y.unique())
    index = np.arange(len(ann))
    trainval_index, test_index = train_test_split(
        index, test_size=0.20, random_state=RANDOM_SEED, stratify=y
    )
    train_index, val_index = train_test_split(
        trainval_index,
        test_size=0.125,
        random_state=RANDOM_SEED,
        stratify=y.iloc[trainval_index],
    )
    for part_name, part_index in (("train", train_index), ("validation", val_index), ("test", test_index)):
        present = set(y.iloc[part_index])
        if present != set(labels):
            raise AssertionError(f"{task} {part_name} is missing a class: {set(labels) - present}")
        split_rows.append({"task": task, "split": part_name, "n_rows": int(len(part_index))})

    x_train = ann.loc[train_index, "model_text"]
    x_val = ann.loc[val_index, "model_text"]
    y_train = y.iloc[train_index]
    y_val = y.iloc[val_index]
    emb_train = embeddings[ann.loc[train_index, "emb_row"]]
    emb_val = embeddings[ann.loc[val_index, "emb_row"]]

    task_scores = []
    for model_name in CLASSICAL_MODEL_NAMES:
        pipeline = make_classical_pipeline(model_name, RANDOM_SEED)
        pipeline.fit(x_train, y_train)
        scored = score_labels(y_val, pipeline.predict(x_val), labels)
        task_scores.append({"task": task, "model": model_name, **scored})
        print(task, model_name, round(scored["macro_f1"], 4))

    scaler = StandardScaler()
    embedding_clf = make_embedding_estimator(RANDOM_SEED)
    embedding_clf.fit(scaler.fit_transform(emb_train), y_train)
    embedding_scores = score_labels(y_val, embedding_clf.predict(scaler.transform(emb_val)), labels)
    task_scores.append({"task": task, "model": EMBEDDING_MODEL_NAME, **embedding_scores})
    print(task, EMBEDDING_MODEL_NAME, round(embedding_scores["macro_f1"], 4))

    score_frame = pd.DataFrame(task_scores).sort_values(
        ["macro_f1", "weighted_f1", "model"], ascending=[False, False, True]
    ).reset_index(drop=True)
    validation_rows.extend(score_frame.to_dict("records"))
    winner_name = score_frame.loc[0, "model"]

    if winner_name == EMBEDDING_MODEL_NAME:
        test_scaler = StandardScaler()
        test_clf = make_embedding_estimator(RANDOM_SEED)
        test_clf.fit(test_scaler.fit_transform(emb_train), y_train)
        y_test_pred = test_clf.predict(test_scaler.transform(embeddings[ann.loc[test_index, "emb_row"]]))
    else:
        test_pipeline = make_classical_pipeline(winner_name, RANDOM_SEED)
        test_pipeline.fit(x_train, y_train)
        y_test_pred = test_pipeline.predict(ann.loc[test_index, "model_text"])

    y_test = y.iloc[test_index]
    held_out = score_labels(y_test, y_test_pred, labels)
    test_rows.append({
        "task": task,
        "selected_model": winner_name,
        "validation_macro_f1": float(score_frame.loc[0, "macro_f1"]),
        "validation_weighted_f1": float(score_frame.loc[0, "weighted_f1"]),
        **{f"test_{key}": value for key, value in held_out.items()},
        "test_used_for_selection": False,
    })
    report = classification_report(
        y_test, y_test_pred, labels=labels, output_dict=True, zero_division=0
    )
    reports[task] = report
    matrix = confusion_matrix(y_test, y_test_pred, labels=labels)
    pd.DataFrame(matrix, index=labels, columns=labels).to_csv(OUT / f"test_confusion_{task}.csv")
    selected[task] = {
        "model": winner_name,
        "validation_macro_f1": float(score_frame.loc[0, "macro_f1"]),
        "test_macro_f1": held_out["macro_f1"],
        "n_classes": len(labels),
        "classes": labels,
    }
    print("SELECTED", task, winner_name, "test macro-F1", round(held_out["macro_f1"], 4))

pd.DataFrame(validation_rows).to_csv(OUT / "validation_scores.csv", index=False)
pd.DataFrame(test_rows).to_csv(OUT / "heldout_test_scores.csv", index=False)
pd.DataFrame(split_rows).to_csv(OUT / "split_sizes.csv", index=False)
(OUT / "selected_models.json").write_text(json.dumps({
    "seed": RANDOM_SEED,
    "split": "stratified 70 percent train, 10 percent validation, 20 percent test",
    "selection_metric": "validation macro-F1",
    "test_used_for_selection": False,
    "model_text": "comment_whitespace_normalized from the raw working corpus",
    "embedding_model": "paraphrase-multilingual-MiniLM-L12-v2",
    "language_limitation": "Embedding scores do not establish that the multilingual model understands Somali.",
    "tasks": selected,
}, indent=2), encoding="utf-8")
(OUT / "heldout_classification_reports.json").write_text(json.dumps(reports, indent=2), encoding="utf-8")
print(pd.DataFrame(test_rows).to_string(index=False))


policy_relevance multinomial_nb 0.4485
policy_relevance complement_nb 0.5393


policy_relevance logistic_regression 0.5195
policy_relevance linear_svc 0.4907


policy_relevance embedding_logistic 0.419
SELECTED policy_relevance complement_nb test macro-F1 0.4914
feedback_type multinomial_nb 0.1964


feedback_type complement_nb 0.2406


feedback_type logistic_regression 0.3114
feedback_type linear_svc 0.2476


...

topic logistic_regression 0.3927
topic linear_svc 0.3712


topic embedding_logistic 0.213
SELECTED topic complement_nb test macro-F1 0.4102


sentiment multinomial_nb 0.495
sentiment complement_nb 0.5468
sentiment logistic_regression 0.5455


sentiment linear_svc 0.5201


sentiment embedding_logistic 0.3992
SELECTED sentiment complement_nb test macro-F1 0.5001
            task      selected_model  validation_macro_f1  validation_weighted_f1  test_macro_f1  test_weighted_f1  test_accuracy  test_used_for_selection
policy_relevance       complement_nb             0.539327                0.734146       0.491360          0.702726       0.733333                    False
   feedback_type logistic_regression             0.311431                0.389930       0.419918          0.474333       0.475000                    False
           topic       complement_nb             0.508698                0.485228       0.410233          0.460297       0.475000                    False
       sentiment       complement_nb             0.546790                0.634406       0.500135          0.588778       0.612500                    False


### Interpretation

Each task uses a stratified 70/10/20 split and seed 42. Complement naive Bayes has the highest validation macro-F1 for `policy_relevance` (0.539), `topic` (0.509), and `sentiment` (0.547). Logistic regression wins `feedback_type` (0.311). The test macro-F1 values, computed after selection, are 0.491, 0.420, 0.410, and 0.500 in that task order. The test split was not used to choose the model. The embedding classifier is the lowest or near-lowest score on every task.


## What this notebook does not claim

The validation comparison chooses the model. The test table is the held-out performance of that chosen model. A later notebook refits the chosen model on all 1,199 labeled comments and predicts the 5,292 comments. Those corpus predictions are a separate pass. They are model output, and the agreement between a refit prediction and a manual label is not a substitute for the test score above.

No class was merged. `other`, `complaint`, and `negative` remain the largest classes in their tasks, so macro-F1 stays the selection metric.
